# Run `coincidence_counters_acquisition.py`
This notebook helps you run the acquisition script, pass custom arguments, run it in the background, and (optionally) live-plot the resulting CSV.

In [1]:
# 🔎 Check the script is present next to this notebook
import os, sys
script_path = 'coincidence_counters_acquisition.py'
if not os.path.exists(script_path):
    raise FileNotFoundError(f'{script_path} not found in: {os.getcwd()}')
print('Found script:', os.path.abspath(script_path))

Found script: c:\Users\Experiment\Documents\Python\VQ_Instruments\ID1000\coincidence_counters_acquisition.py


In [2]:
# ▶️ Run with default arguments (uses values defined in the script)
!python coincidence_counters_acquisition.py

## Run with custom arguments
Adjust interval, duration, IP address, output file, coincidence window, integration time, and log path as needed.

In [6]:
# 🛠️ Customize and run
CUSTOM_INTERVAL = 2            # seconds
CUSTOM_DURATION = 20           # seconds (0 or None for infinite)
TC_ADDRESS = '169.254.224.106' # device IP
COUNTERS_FILE = 'counters_output.csv'
COINCIDENCE_WINDOW_PS = 5000   # picoseconds
INTEGRATION_NS = 1000          # nanoseconds (0 for endless accumulation)
LOG_PATH = 'acquisition_log.txt'
VERBOSE = True

cmd = (
    f"python coincidence_counters_acquisition.py"
    f" --interval {CUSTOM_INTERVAL}"
    f" --duration {CUSTOM_DURATION}"
    f" --address {TC_ADDRESS}"
    f" --file '{COUNTERS_FILE}'"
    f" --window {COINCIDENCE_WINDOW_PS}"
    f" --integration {INTEGRATION_NS}"
    f" --log-path '{LOG_PATH}'"
    f"{' --verbose' if VERBOSE else ''}"
)
print('Running:', cmd)
!{cmd}

Running: python coincidence_counters_acquisition.py --interval 2 --duration 20 --address 169.254.224.106 --file 'counters_output.csv' --window 5000 --integration 1000 --log-path 'acquisition_log.txt' --verbose


## Run in the background
Use this if you want to keep working in the notebook while the script runs.

In [9]:
# 🧵 Start background process
import subprocess
bg_args = [
    'python', 'coincidence_counters_acquisition.py',
    '--interval', '3',
    '--duration', '60',
    '--file', 'counters_output.csv',
]
process = subprocess.Popen(bg_args)
print('Started PID:', process.pid)

Started PID: 18520


In [10]:
# ⛔ Stop a background process by PID (set the PID printed above)
import os, signal
PID_TO_STOP = None  # <- put PID here, e.g. 12345
if PID_TO_STOP is not None:
    os.kill(PID_TO_STOP, signal.SIGTERM)
    print('Sent SIGTERM to', PID_TO_STOP)
else:
    print('Set PID_TO_STOP to a valid PID to stop the process.')

Set PID_TO_STOP to a valid PID to stop the process.


## Live plot of counters CSV (optional)
This cell will refresh every few seconds and plot the CSV file if present.

In [11]:
# 📈 Live plotter
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

CSV_PATH = 'counters_output.csv'  # match --file above or the script default
REFRESH_EVERY = 3                 # seconds
MAX_POINTS = None                 # e.g. 500 to limit

def plot_csv(path):
    if not os.path.exists(path):
        print(f'Waiting for {path} ...')
        return False
    try:
        df = pd.read_csv(path, sep=';')
    except Exception as e:
        print('CSV not ready yet:', e)
        return False
    if 'time' not in df.columns:
        print('CSV missing 'time' column; header not written yet?')
        return False
    if MAX_POINTS:
        df = df.tail(MAX_POINTS)
    plt.figure(figsize=(10,5))
    for col in df.columns:
        if col == 'time':
            continue
        plt.plot(df['time'], df[col], label=col)
    plt.xlabel('time (s)')
    plt.ylabel('counts')
    plt.title('Coincidence & input counters vs. time')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    display(plt.gcf())
    plt.close()
    return True

try:
    while True:
        clear_output(wait=True)
        ok = plot_csv(CSV_PATH)
        time.sleep(REFRESH_EVERY)
except KeyboardInterrupt:
    print('Stopped live plotting.')

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3377272435.py, line 21)